# Silver Layer Exploration
Cleaned, deduplicated, and enriched ride events. This is the layer used as the
input for Gold aggregations and ML training.

In [ ]:
import sys
sys.path.insert(0, '..')

from storage.spark_session import build_delta_spark
from storage.delta_config import SILVER_RIDES_TABLE

spark = build_delta_spark('SilverExploration')

In [ ]:
rides = spark.read.format('delta').load(SILVER_RIDES_TABLE)
print(f'Row count: {rides.count():,}')
rides.printSchema()

In [ ]:
rides.show(5)

## Data Quality Checks

In [ ]:
from pyspark.sql import functions as F

# Null check on key columns
null_report = rides.select([
    F.sum(F.col(c).isNull().cast('int')).alias(c)
    for c in ['ride_id', 'event_ts', 'city_zone', 'surge_multiplier', 'gross_fare_inr']
])
null_report.show()

In [ ]:
# Surge bounds check — should all be in [1.0, 3.5]
out_of_bounds = rides.filter((F.col('surge_multiplier') < 1.0) | (F.col('surge_multiplier') > 3.5)).count()
print(f'Rows with surge out of [1.0, 3.5]: {out_of_bounds}')

## Ride Volume by Hour

In [ ]:
hourly = (
    rides
    .groupBy('event_hour')
    .agg(
        F.count('ride_id').alias('ride_count'),
        F.round(F.avg('surge_multiplier'), 2).alias('avg_surge'),
        F.round(F.sum('gross_fare_inr'), 2).alias('total_revenue_inr'),
    )
    .orderBy('event_hour')
    .toPandas()
)

hourly

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.bar(hourly['event_hour'], hourly['ride_count'], color='steelblue')
ax1.set_title('Ride Volume by Hour of Day')
ax1.set_xlabel('Hour')
ax1.set_ylabel('Ride Count')
ax1.axvspan(7, 10, alpha=0.2, color='orange', label='Morning peak')
ax1.axvspan(17, 21, alpha=0.2, color='red', label='Evening peak')
ax1.legend()

ax2.plot(hourly['event_hour'], hourly['avg_surge'], marker='o', color='crimson')
ax2.set_title('Average Surge Multiplier by Hour')
ax2.set_xlabel('Hour')
ax2.set_ylabel('Avg Surge')
ax2.axhline(y=1.0, linestyle='--', color='gray')

plt.tight_layout()
plt.show()

## Revenue by Zone

In [ ]:
zone_revenue = (
    rides
    .groupBy('city_zone')
    .agg(
        F.count('ride_id').alias('rides'),
        F.round(F.sum('gross_fare_inr'), 2).alias('revenue_inr'),
        F.round(F.avg('surge_multiplier'), 3).alias('avg_surge'),
    )
    .orderBy('revenue_inr', ascending=False)
    .toPandas()
)

zone_revenue.plot(kind='bar', x='city_zone', y='revenue_inr', title='Revenue by Zone (INR)', figsize=(10, 5))
plt.tight_layout()
plt.show()

zone_revenue